# Air Quality Prediction — PM2.5 forecasting

**Goal:** given the last few hours of pollution and weather readings, predict the PM2.5 concentration one hour ahead, and translate it into an AQI category.

This notebook walks through the same pipeline that lives in `src/`. Run the cells top to bottom.

**Contents**
1. Setup and data loading
2. Exploratory data analysis
3. Feature engineering
4. Baseline and models
5. Evaluation
6. Making a forecast
7. What to try next

## 1. Setup and data loading

In [ ]:
import sys, warnings
sys.path.append("../src")   # use "src" if you run this from the project root
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 4)
pd.set_option("display.width", 120)

In [ ]:
from data import load_raw

df = load_raw()
print(df.shape)
df.head()

In [ ]:
df.describe().round(2)

### What each column means

| column | meaning |
|---|---|
| `pm25` | PM2.5 concentration, ug/m3 — **this is what we predict** |
| `temp` | temperature, °C |
| `dew_point` | dew point, °C (a humidity proxy) |
| `pressure` | atmospheric pressure, hPa |
| `wind_dir` | combined wind direction (categorical) |
| `wind_speed` | cumulative wind speed |
| `rain_hours`, `snow_hours` | cumulative hours of rain/snow |

## 2. Exploratory data analysis

Before modelling, understand the shape of the problem.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
df["pm25"].resample("D").mean().plot(ax=ax)
ax.set(title="Daily mean PM2.5", ylabel="ug/m3")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

df["pm25"].plot.hist(bins=80, ax=axes[0], title="Distribution (right-skewed)")
df.groupby(df.index.hour)["pm25"].mean().plot(ax=axes[1], marker="o", title="Average by hour of day")
df.groupby(df.index.month)["pm25"].mean().plot(ax=axes[2], marker="o", title="Average by month")
plt.tight_layout(); plt.show()

Two things to notice, and both drive the design of the features:

- **Right-skewed target.** Most hours are ordinary; a few are extreme. Models trained on squared error will chase the spikes, and that's where most of the error will live.
- **Strong daily and seasonal cycles.** The hour and the month carry real signal, so they become features.

In [ ]:
# Autocorrelation: how much does the current value tell us about later values?
lags = [1, 2, 3, 6, 12, 24, 48, 72]
for L in lags:
    print(f"lag {L:>3}h  correlation {df['pm25'].corr(df['pm25'].shift(L)):.3f}")

Correlation at lag 1 is very high. That's the single most useful fact in this project: **the last reading is already a good prediction.** Any model has to beat that, which is exactly what the persistence baseline below measures.

In [ ]:
corr = df.drop(columns=["wind_dir"]).corr()["pm25"].sort_values()
corr.drop("pm25").plot.barh(title="Correlation of weather variables with PM2.5")
plt.show()

## 3. Feature engineering

Every feature must be computable at prediction time. Lags and rolling windows are shifted so no future information leaks in.

In [ ]:
from features import build_dataset, chronological_split, LAGS, ROLLING_WINDOWS
from config import TEST_FRACTION

print("lags:", LAGS)
print("rolling windows:", ROLLING_WINDOWS)

X, y, baseline = build_dataset(df)
X_train, X_test, y_train, y_test, base_test = chronological_split(X, y, baseline, TEST_FRACTION)

print(f"\n{X.shape[1]} features, {len(X):,} rows")
print(f"train {X_train.index.min()} -> {X_train.index.max()}")
print(f"test  {X_test.index.min()} -> {X_test.index.max()}")
X.columns.tolist()

**Why the split is chronological.** `train_test_split(shuffle=True)` would put hours from the same day in both sets. The model would be interpolating between points it has already seen, and the test score would be a lie. Train on the past, test on the future.

## 4. Baseline and models

The baseline is *persistence*: predict that the next hour equals the current hour. It's free and it's hard to beat.

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

def score(name, y_true, y_pred):
    return {
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }

rows = [score("persistence (baseline)", y_test, base_test)]
pd.DataFrame(rows).round(3)

In [ ]:
from train import get_models

fitted = {}
for name, model in get_models().items():
    model.fit(X_train, y_train)
    fitted[name] = model
    rows.append(score(name, y_test, model.predict(X_test)))

results = pd.DataFrame(rows).set_index("model").round(3)
results

Read this table honestly. If a model's RMSE is only a few percent below the baseline, the extra complexity is buying very little — and saying so is a better finding than pretending otherwise. Gains usually come from the lag features, not from swapping one tree ensemble for another.

## 5. Evaluation

A single number hides where the model fails. Look at the errors.

In [ ]:
best_name = results.drop("persistence (baseline)")["RMSE"].idxmin()
best = fitted[best_name]
pred = pd.Series(best.predict(X_test), index=y_test.index)
print("best:", best_name)

window = slice(0, 24 * 10)
fig, ax = plt.subplots(figsize=(13, 4))
y_test.iloc[window].plot(ax=ax, label="Actual", lw=1.6)
pred.iloc[window].plot(ax=ax, label="Predicted", lw=1.4, alpha=.85)
ax.set(title="First 10 days of the test set", ylabel="PM2.5"); ax.legend()
plt.show()

In [ ]:
resid = pred - y_test

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(y_test, pred, s=4, alpha=.2)
lim = float(y_test.max())
axes[0].plot([0, lim], [0, lim], "r--"); axes[0].set(xlabel="Actual", ylabel="Predicted", title="Predicted vs actual")
axes[1].hist(resid, bins=60); axes[1].axvline(0, color="r", ls="--"); axes[1].set(title="Residuals")
axes[2].scatter(y_test, resid, s=4, alpha=.2); axes[2].axhline(0, color="r", ls="--")
axes[2].set(xlabel="Actual", ylabel="Residual", title="Error vs pollution level")
plt.tight_layout(); plt.show()

The third panel is the important one: **error grows with the pollution level, and predictions lag behind sudden spikes.** That's the honest limitation of this model — it is good at ordinary hours and weakest exactly when a forecast would matter most.

In [ ]:
from sklearn.inspection import permutation_importance

r = permutation_importance(best, X_test[:3000], y_test[:3000], n_repeats=5, random_state=0, n_jobs=-1)
imp = pd.Series(r.importances_mean, index=X_test.columns).sort_values().tail(15)
imp.plot.barh(title="Permutation importance"); plt.show()

## 6. Making a forecast

Recursive multi-step: predict t+1, feed it back in, predict t+2, and so on.

In [ ]:
from aqi import pm25_to_aqi, advice

for c in [25, 55, 85, 140, 260]:
    a, label = pm25_to_aqi(c)
    print(f"{c:>4} ug/m3 -> AQI {a:>3}  {label}")

In [ ]:
# Uses the model saved by `python src/train.py`
from predict import forecast

f = forecast(hours=6)
for ts, v in f.items():
    a, label = pm25_to_aqi(v)
    print(f"{ts:%Y-%m-%d %H:%M}  {v:7.1f} ug/m3   AQI {a:>3}  {label}")

## 7. What to try next

In rough order of how much they'll teach you:

1. **Predict further ahead.** Set `HORIZON = 24` in `config.py` and re-run. Watch the model collapse toward the seasonal average — that gap between 1-hour and 24-hour prediction is the real lesson of the project.
2. **Train on `log1p(pm25)`** and invert with `expm1`. The target is right-skewed; this often helps at the high end where the model is currently worst.
3. **Proper time-series cross-validation** with `TimeSeriesSplit(n_splits=5)` instead of one split, so your score isn't an accident of which months landed in the test set.
4. **Classify AQI category** instead of regressing the number. A confusion matrix over Good/Moderate/Poor/Severe is easier to act on, and precision on the "Severe" class is the metric that would actually matter.
5. **Compare against a classical model** — SARIMA or Prophet — to see what the tree ensemble is and isn't adding.
6. **Try a sequence model** (LSTM or a 1D CNN) on raw windows. Worth doing to find out that on this size of tabular data, gradient boosting is usually still competitive.
7. **Use real local data.** The CPCB CAAQMS portal publishes Indian station data, and OpenAQ has a free API. Swapping the data source only means editing `src/data.py`.